In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.tree import DecisionTreeClassifier

In [ ]:
df = pd.read_csv('/content/Titanic-Dataset.csv')

In [ ]:
df.head(2)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C


In [ ]:
df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [ ]:
# Missing Values in Age and Embarked columns

   #Pipeline:
   # Step 1 -> SimpleImpute on Age column
   # Step 2 -> OHE on Sex and Embarked columns
   # Step 3 -> Scaling all the columns values
   # Step 4 -> Feature Selection (Top 5)
   # Step 5 -> Train model using DecisionTree


In [ ]:
df.drop(columns = ['PassengerId', 'Name', 'Ticket', 'Cabin'], inplace = True)

In [ ]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns = ['Survived']),
                                                    df['Survived'],
                                                    test_size = 0.2,
                                                    random_state = 42)

In [ ]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


In [ ]:
y_train.head()

,Survived
331,0
733,0
382,0
704,0
813,0


In [ ]:
# imputation transformer
trf1 = ColumnTransformer([
    ('impute_age', SimpleImputer(), ['Age']),
    ('impute_embarled', SimpleImputer(strategy = 'most_frequent'), ['Embarked'])
], remainder = 'passthrough'
)

In [ ]:
trf1

ColumnTransformer(remainder='passthrough',
                  transformers=[('impute_age', SimpleImputer(), [2]),
                                ('impute_embarled',
                                 SimpleImputer(strategy='most_frequent'),
                                 [6])])

In [ ]:
# OneHotEncoding
trf2 = ColumnTransformer([
    ('ohe_sex_embarked', OneHotEncoder(sparse_output = False), ['Sex', 'Embarked'])
])

In [ ]:
trf2

ColumnTransformer(remainder='passthrough',
                  transformers=[('ohe_sex_embarked', OneHotEncoder(), [1, 6])])

In [ ]:
# Scaling
trf3 = ColumnTransformer([
    ('scale', MinMaxScaler(), slice(0,10))
])

In [ ]:
trf3

ColumnTransformer(transformers=[('scale', MinMaxScaler(), slice(0, 10, None))])

In [ ]:
# feature selection
trf4 = SelectKBest(score_func = chi2, k = 5)

In [ ]:
# train the model
trf5 = DecisionTreeClassifier()

# Create Pipeline

In [ ]:
pipe = Pipeline([
    ('trf1', trf1),
    ('trf2', trf2),
    ('trf3', trf4), # Swap trf3 and trf4
    ('trf4', trf3), # Swap trf3 and trf4
    ('trf5', trf5)
])

In [ ]:
# train
pipe.fit(X_train, y_train)

ValueError: could not convert string to float: 'male'